# Type B Bounding-Box Prompt Test

This notebook tests one vision model against `tests/orange_circles.png` using the same prompt generator and box parser as the Type B benchmarks. The target class is `orange circle`.

In [ ]:
from pathlib import Path
import sys

from PIL import Image, ImageDraw
import matplotlib.pyplot as plt

candidate_roots = [Path.cwd(), *Path.cwd().parents]
REPO_ROOT = next(
    (path for path in candidate_roots if (path / "benchmarks").is_dir() and (path / "models").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Run this notebook from inside the repository.")
REPO_ROOT = REPO_ROOT.resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

IMAGE_PATH = REPO_ROOT / "tests" / "orange_circles.png"
TARGET_CLASS = "orange circle"

image = Image.open(IMAGE_PATH).convert("RGB")
display(image)
print(f"Image: {IMAGE_PATH}")
print(f"Size: {image.size[0]} x {image.size[1]}")

## Build the production Type B prompt

In [ ]:
from benchmarks.detection import DetectionBenchmark


class _PromptOnlyDataset:
    name = "orange_circles"
    labels = [TARGET_CLASS]

    @staticmethod
    def normalize_text(text):
        return str(text).strip().lower()


benchmark = DetectionBenchmark(dataset=_PromptOnlyDataset(), name="orange_circles")
prompt = benchmark.make_prompt([TARGET_CLASS])
print(prompt)

## Load one model

The default is the repository's `GPT4` wrapper (`gpt-4o`). It reads `OPENAI_API_KEY` from the environment.

In [ ]:
from models import GPT4

model = GPT4(max_new_tokens=128, temperature=0.0)
print(f"Model: {model.name}")

## Run the single prediction

In [ ]:
prediction = model.predict(image, prompt)
print("Raw model response:")
print(prediction)

## Parse and visualize the returned boxes

In [ ]:
normalized_boxes = benchmark.get_predicted_boxes(prediction)
pixel_boxes = benchmark.postprocess_predicted_boxes(normalized_boxes, image=image)

print(f"Parsed boxes: {len(pixel_boxes)}")
for index, box in enumerate(pixel_boxes, start=1):
    print(f"{index}: {box['xyxy']}")
assert len(pixel_boxes) == 2, f"Expected 2 orange-circle boxes, received {len(pixel_boxes)}"

annotated = image.copy()
draw = ImageDraw.Draw(annotated)
line_width = max(2, round(max(image.size) / 250))
for box in pixel_boxes:
    draw.rectangle(box["xyxy"], outline="red", width=line_width)

plt.figure(figsize=(10, 8))
plt.imshow(annotated)
plt.axis("off")
plt.title(f"{model.name}: {len(pixel_boxes)} parsed box(es)")
plt.show()